# Data Cleaning (Puripat, wk1)

โน๊ตบุ๊คของ Puripat ทำ cleaning อิสระจาก Saphondanai แล้วเทียบผลท้ายไฟล์ ก่อนรวมเป็น [`../src/clean_pipeline.py`](../src/clean_pipeline.py)
ตาม [TASKS.md](../TASKS.md) และตรวจ leakage ตาม [README 6.2](../README.md#62-data-leakage-guard)

ขั้นตอน: 1) โหลด + สำรวจ 2) missing/duplicate 3) ตัดคอลัมน์ noise 4) ตรวจ leakage 5) encode 6) เทียบกับ pipeline/Saphondanai

In [ ]:
import pandas as pd
import sys
sys.path.append("../src")  # ให้ import clean_pipeline ได้ตอนเทียบผล
from clean_pipeline import clean_data

raw = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(raw.shape)
raw.head()

## 1. สำรวจข้อมูลเบื้องต้น

In [ ]:
print(raw.dtypes.value_counts())
print("Attrition rate:", (raw["Attrition"] == "Yes").mean().round(3))
raw["Attrition"].value_counts()

## 2. Missing value และแถวซ้ำ

In [ ]:
print("missing รวม:", raw.isna().sum().sum())
print("แถวซ้ำ:", raw.duplicated().sum())
print("EmployeeNumber ซ้ำ:", raw["EmployeeNumber"].duplicated().sum())

ไม่มี missing และไม่มีแถวซ้ำ จึงไม่ต้อง impute หรือ drop แถว

## 3. ตัดคอลัมน์ noise (ค่าคงที่ / ตัวระบุแถว)

In [ ]:
const_cols = [c for c in raw.columns if raw[c].nunique() == 1]
print("ค่าคงที่ทุกแถว:", const_cols)
print("EmployeeNumber unique =", raw["EmployeeNumber"].nunique(), "จาก", len(raw), "แถว -> ตัวระบุแถว")
df = raw.drop(columns=const_cols + ["EmployeeNumber"])
df.shape

## 4. ตรวจ Data Leakage (README 6.2)

ทุกฟีเจอร์ที่เหลือเป็นข้อมูล snapshot ตอนที่พนักงานยังทำงานอยู่ (ข้อมูลส่วนตัว, ตำแหน่ง, รายได้, ผลสำรวจความพึงพอใจ, อายุงาน)
ไม่มีฟีเจอร์ที่เกิดหลังการตัดสินใจลาออก (เช่น วันสุดท้าย, ผล exit interview) จึงไม่ตัดคอลัมน์ไหนเพิ่มเพราะ leakage

ข้อสังเกต: `YearsSinceLastPromotion`, `YearsAtCompany` ฯลฯ นับถึงวันที่ถ่าย snapshot ถ้าทีมได้ข้อมูลจริงต้องยืนยันวันที่ snapshot กับ HR อีกครั้ง

## 5. Encode

In [ ]:
df["Attrition"] = df["Attrition"].map({"No": 0, "Yes": 1})
df["BusinessTravel"] = df["BusinessTravel"].map({"Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2})  # ordinal
df["OverTime"] = df["OverTime"].map({"No": 0, "Yes": 1})
df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})

nominal = df.select_dtypes("object").columns.tolist()
print("one-hot:", nominal)
df = pd.get_dummies(df, columns=nominal, dtype=int)
print(df.shape, "| object เหลือ:", df.select_dtypes("object").shape[1], "| missing:", df.isna().sum().sum())

## 6. เทียบกับ `clean_pipeline.clean_data` และ CSV ของ Saphondanai

In [ ]:
pipe = clean_data(raw)
saph = pd.read_csv("../data/processed/attrition_cleaned_S.csv")

print("เหมือน pipeline:", df.shape == pipe.shape and set(df.columns) == set(pipe.columns) and df[pipe.columns].equals(pipe))
print("เหมือนของ Saphondanai:", set(df.columns) == set(saph.columns) and df[saph.columns].equals(saph))
print("คอลัมน์ต่างกัน:", set(df.columns) ^ set(saph.columns))

## สรุป

- 1,470 แถว, ไม่มี missing/ซ้ำ, ตัด 4 คอลัมน์ noise, ไม่มี leakage
- Attrition ~16% (imbalance) ใน wk2–3 จึงต้องใช้ stratified split / class weight
- ผลลัพธ์เหมือนของ Saphondanai (ดู cell ด้านบน) จึงรวมเป็น `src/clean_pipeline.py` ได้